
# Supresvised Learning. Classification

---

**Tathagata Chowdhury**, ISU ID: <your_ISU_ID>

---

Topics: pytorch, data preparation, data augmentation, custom model, transfer learning, classification, segmentation

Основные задачи

1. Решить проблему **классификации** дорожных фотографий
  1. Подготовьте данные и решите проблему несбалансированного набора данных
  2. Реализуйте и обучите пользовательский классификатор
  3. Реализуйте конвейер проверки
  4. Добавьте дополнения к данным и сравните результаты
  5. Загрузка предварительно обученной модели и настройка для фотографий дорог

Подзадачи уточняются ниже.



**Требования**.

1. Вы выполнили лабораторную работу самостоятельно.
2. Выполнены все задачи и подзадачи. Выводы обоснованы и верны.
3. Код выполним, а результаты воспроизводимы. Если данные взяты из личного Google Drive, необходимо указать ссылку на них.
4. Тетрадь Colab должна содержать результаты выполнения кода в выводе ячейки.
5. Основная структура Colab Notebook сохраняется. Вы можете добавлять разделы, писать дополнительные функции и существенно не изменять разделы. Если вы удаляете текст и прочее, убедитесь, что все результаты выполнения задачи остаются явно выделенными.
6. Чтобы сдать лабораторную, необходимо отправить ссылку на общий доступ к Colab Notebook на oaevstafev@itmo.ru


----

Main Tasks
1. Solve the road photo **classification** problem
  1. Prepare data and solve imbalanced dataset problem
  2. Implement and train a custom classifier
  3. Implement validation pipeline
  4. Add data augmentation and compare results
  5. Load pre-trained model and tune for road photos
  
Subtasks are specified in-place.


**Requirements**
1. You did the lab on your own.
2. All tasks and subtasks are completed. Conclusions are well-founded and valid.
3. The code is executable and the results are reproducible. If data is taken from your personal Google Drive, a link to the data should be provided.
4. The Colab notebook must contain the results of the code execution in the cell output.
5. The main structure of this Colab Notebook is preserved. You can add sections, write additional functions, and not significatly change the sections. If you remove any text and other things, make sure that all task results remain explicitly marked.
6. To pass the lab you should send the sharing link to your Colab Notebook at oaevstafev@itmo.ru




 ## Enviroment setup

Import packages

In [ ]:
import numpy as np

import torch
from torch import nn
from torch.nn import functional as F
from torch import optim

from torchvision import models, transforms

import time
import math
import random

from torch.utils.data import TensorDataset, DataLoader, Dataset
from sklearn.metrics import accuracy_score

import seaborn as sns
from matplotlib import colors, pyplot as plt
from matplotlib import rcParams
from IPython.display import clear_output

from tqdm import tqdm

%matplotlib inline
rcParams['figure.figsize'] = (15,4)
sns.set(style="darkgrid", font_scale=1.4)

Define the device variable to run notebook on gpu if it is available and cpu in the opposite case without any notebook modifications

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

GPU = torch.device('cuda:0')
CPU = torch.device('cpu')

Specify a seed to more reproducable results. Now the mentioned below random generators will produce the same random sequencies.

In [ ]:
SEED = 42 # You can set any interger here

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

Here some snippets to check GPU params

In [ ]:
!nvidia-smi
train_on_gpu = torch.cuda.is_available()

if not train_on_gpu:
    print('CUDA is not available.  Training on CPU ...')
else:
    print('CUDA is available!  Training on GPU ...')

The snippet below can be used to check GPU memory allocation and clean it

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

#Additional Info when using cuda
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))
    print('Memory Usage:')
    print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
    print('Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')

## Dataset loading

### Way 1. By direct link

Используйте одну из ссылок / Use one of the links below
- https://www.dropbox.com/s/j5j1kd4h55x4pmp/segmentation.zip?dl=0
- http://mlr.vedyakov.com/segmentation_new.zip

In [ ]:
!wget -c http://mlr.vedyakov.com/segmentation_new.zip -O segmentation.zip

--2024-05-10 20:03:23--  http://mlr.vedyakov.com/segmentation_new.zip
Resolving mlr.vedyakov.com (mlr.vedyakov.com)... 77.234.215.110
Connecting to mlr.vedyakov.com (mlr.vedyakov.com)|77.234.215.110|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 418673165 (399M) [application/zip]
Saving to: ‘segmentation.zip’

segmentation.zip    100%[===================>] 399.28M  3.89MB/s    in 2m 14s  

2024-05-10 20:05:38 (2.97 MB/s) - ‘segmentation.zip’ saved [418673165/418673165]



Then we should unzip

In [ ]:
!unzip -q ./segmentation.zip

### Way 2. From personal Google drive

Add file to your personal google drive https://drive.google.com/file/d/1yqxfvTutEGOFMct5U_zVIP6e9v7R3AcL/view?usp=sharing or download from cloud services (Here are a few links since they stop working periodically)
- https://www.dropbox.com/s/j5j1kd4h55x4pmp/segmentation.zip?dl=0
- https://niuitmo-my.sharepoint.com/:u:/g/personal/vedyakov_niuitmo_ru/Ec4GW6NfTaJOhdUgfHwwTpQB09j7apoB9pJ3tq7sIWLwsg?e=iADkMZ
- https://disk.yandex.ru/d/azhOrDCv1P3rDw
and upload to the personal google drive. In the example, the file in the University/MLR directory.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

In [ ]:
!ls /content/gdrive/MyDrive/University/MLR

In [ ]:
!unzip -q /content/gdrive/MyDrive/University/MLR/segmentation.zip

## Data preparation

### Data checking

In [ ]:
X = np.load('./x_train.npy')
Y = np.load('./y_train.npy')

print(X.shape)
print(Y.shape)
print(Y[0])

In [ ]:
plt.figure(figsize=(18, 6))
for i in range(6):
    j = 100+10*i;
    plt.subplot(2, 6, i+1)
    plt.axis("off")
    plt.imshow(X[j])

    plt.subplot(2, 6, i+7)
    plt.axis("off")
    plt.imshow(Y[j].squeeze(), cmap='gray')
plt.show();

### Preparing target values for the classification task

Let us find all possible Y values in the dataset

In [ ]:
unique, counts = np.unique(Y, return_counts=True)

result = np.column_stack((unique, counts))
print(result)

If number of non-black pixels is less than threshold, then we say that there is not a road sign on the photo

In [ ]:
sum_by_image = np.sum(Y, (1, 2, 3))
print(sum_by_image.shape)

In [ ]:
unique, counts = np.unique(sum_by_image, return_counts=True)
result = np.column_stack((unique, counts))
print (result)

In [ ]:
threshold = 0  # any non-black pixel in the mask means a road sign is present
Y_c = np.array(sum_by_image > threshold, dtype='uint8')
print(Y_c.shape)

Now we have labels for classification task

In [ ]:
print(Y_c[0:100])

In [ ]:
plt.figure(figsize=(18, 6))
for i in range(6):
    j = 2 + i;
    plt.subplot(2, 6, i+1)
    plt.axis("off")
    plt.imshow(X[j])

    plt.subplot(2, 6, i+7)
    plt.axis("off")
    plt.imshow(Y[j].squeeze(), cmap='gray')
plt.show();

### Data normalization

For pytorch, the channel should be the second dimension, not the last one.

In [ ]:
X_r = np.transpose(X, axes=(0, 3, 1, 2))
Y_r = Y_c
print(X_r.shape, Y_r.shape)

Normalize data and convert to tensors

In [ ]:
X_f = np.array(X_r / 255, dtype='float32')  # convert 0-255 values to 0-1
X_t = torch.FloatTensor(X_f)

# standardize each channel with ImageNet statistics (same stats used by the
# pretrained transfer-learning model later, so all models share one input scale)
for x in X_t:
  transforms.functional.normalize(x, (0.485, 0.456, 0.406), (0.229, 0.224, 0.225), inplace=True)

Y_t = torch.FloatTensor(Y_r)

Splitting dataset

Let us divide for traning, validation and test correspondingly.

In [ ]:
ix = np.random.choice(len(X_t), len(X_t), False)
n = len(X_t)
val_index = int(0.70 * n)   # 70% train
test_index = int(0.85 * n)  # 15% validation, 15% test
tr, val, ts = np.split(ix, [val_index, test_index])
print('train/val/test sizes:', len(tr), len(val), len(ts))

In [ ]:
X_train_t = X_t[tr]
X_val_t = X_t[val]
X_test_t = X_t[ts]

Y_train_t = Y_t[tr]
Y_val_t = Y_t[val]
Y_test_t = Y_t[ts]

train_dataset = TensorDataset(X_train_t, Y_train_t)
val_dataset = TensorDataset(X_val_t, Y_val_t)
test_dataset = TensorDataset(X_test_t, Y_test_t)

In [ ]:
unique, counts = np.unique(Y_train_t, return_counts=True)
result = np.column_stack((unique, counts))
print (result)

Check converted and normalized data

In [ ]:
def imshow(inp, title=None, plt_ax=plt, default=False):
    inp = inp.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    plt_ax.imshow(inp)
    if title is not None:
      if hasattr(plt_ax, 'set_title'):
        plt_ax.set_title(title)
      if hasattr(plt_ax, 'title'):
        plt_ax.title(title)
    plt_ax.grid(False)

In [ ]:
imshow(X_train_t[0])

### Classes balancing and creating dataloaders

Let's count how many pictures we have for each class.

In [ ]:
unique, counts = np.unique(Y_r, return_counts=True)
result = np.column_stack((unique, counts))
print(result)

The dataset is not balanced, so let's define the weights for each class.

In order for the classes to be balanced in training, we calculate the weights and create a sampler.
For the validation and test, we will not use balancing.

In [ ]:
unique, counts = np.unique(Y_r[tr], return_counts=True)
# inverse-frequency weights: rarer class gets a larger weight
w = counts.sum() / (len(counts) * counts)
print(w)

In [ ]:
from torch.utils.data import WeightedRandomSampler

weights = [w[0] if x == 0 else w[1] for x in Y_r[tr]]
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

print(Y_r[tr][50:100:5])
print(weights[50:100:5])

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=128, sampler=sampler)
val_dataloader = DataLoader(val_dataset, batch_size=128, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False)

Let's check that the classes are balanced now

In [ ]:
YY_check = np.array([])

for _, y in train_dataloader:
  YY_check = np.hstack((YY_check, y.detach().numpy()))

unique, counts = np.unique(YY_check, return_counts=True)
result = np.column_stack((unique, counts))
print(result)

Taking into account the imbalance of classes in the validation and test samples, we will use a special metric to evaluate the quality of training.

In [ ]:
from sklearn.metrics import balanced_accuracy_score

## Custom classifier and training

**Pass requirement**: balanced_accuracy_score metric on the test part of the dataset is at least **0.95**.

In [ ]:
class SimpleCnn(nn.Module):

    def __init__(self, size, n_classes):
        super().__init__()

        # 4 conv blocks, each Conv->ReLU->MaxPool(2) -> spatial size /16 total.
        # Channels grow 3 -> 16 -> 32 -> 64 -> 64.
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16),
            nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32),
            nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),
            nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.conv4 = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64),
            nn.ReLU(inplace=True), nn.MaxPool2d(2))

        fc_inputs = int((size/16)*(size/16)*64)

        # binary classification -> single logit (used with BCEWithLogitsLoss)
        self.out = nn.Sequential(
            nn.Linear(fc_inputs, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 1 if n_classes == 2 else n_classes),
        )


    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)

        x = x.view(x.size(0), -1)
        logits = self.out(x)

        return logits.squeeze()

Let us define training function

In [ ]:
def train(model, optimizer, loss_function, dataloader, max_epochs = 10):
  losses = np.zeros(max_epochs)

  model.train()
  for epoch in range(max_epochs):
      for it, (X_batch, y_batch) in enumerate(dataloader):
          X_batch = X_batch.to(device)
          y_batch = y_batch.to(device)

          optimizer.zero_grad()

          outp = model(X_batch)  # predict y with model using X_batch

          loss = loss_function(outp.squeeze(), y_batch)
          losses[epoch] = losses[epoch] + loss.detach().flatten()[0]

          loss.backward()        # back propagation and gradient calculation
          optimizer.step()       # weights update

          y_batch = y_batch.cpu()
          X_batch = X_batch.cpu()
          outp = outp.cpu()

      print(f"Epoch {epoch}: loss={losses[epoch]}")

  return losses

Now we can create model, define loss function and optimizer

In [ ]:
size = X_t.shape[-1]  # image side length (square images)
simpleNN = SimpleCnn(size, n_classes=2).to(device)  # create model and move it to device
loss_function = nn.BCEWithLogitsLoss()              # binary classification on a single logit
learning_rate = 1e-3
optimizer = torch.optim.Adam(simpleNN.parameters(), lr=learning_rate)

and train our model

In [ ]:
max_epochs = 15
losses = train(simpleNN, optimizer, loss_function, train_dataloader, max_epochs)

In [ ]:
def plot_losses(losses):
  plt.figure(figsize=(12, 8))
  plt.plot(range(len(losses)), losses)
  plt.xlabel("Iteration")
  plt.ylabel("Loss")
  plt.show()

In [ ]:
plot_losses(losses)

We can additionally train the model with the lowel learning rate. However, the better idea is to use [schedulers](https://www.kaggle.com/code/isbhargav/guide-to-pytorch-learning-rate-scheduling), so we suggest to try it.

In [ ]:
# [OPTIONAL] extra fine-tuning with a smaller learning rate via a scheduler
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
extra_epochs = 5
simpleNN.train()
for epoch in range(extra_epochs):
    running = 0.0
    for X_batch, y_batch in train_dataloader:
        X_batch = X_batch.to(device); y_batch = y_batch.to(device)
        optimizer.zero_grad()
        outp = simpleNN(X_batch)
        loss = loss_function(outp.squeeze(), y_batch)
        loss.backward(); optimizer.step()
        running += loss.item()
        X_batch = X_batch.cpu(); y_batch = y_batch.cpu()
    scheduler.step()
    print(f"extra epoch {epoch}: loss={running:.4f}, lr={scheduler.get_last_lr()[0]}")

## Model validation

In order to check overfitting we compate metric results on the train and validation datasets. If everything is ok, then we check on test dataset.

In [ ]:
@torch.no_grad()
def predict(dataloader, model):
    model.eval()
    predictions = np.array([])
    y_target = np.array([])
    for x_batch, y_batch in dataloader:
        y_target = np.hstack((y_target, y_batch.numpy().flatten()))
        x_batch = x_batch.to(device)
        lgts = model(x_batch)
        probs = torch.sigmoid(lgts)
        preds = (probs > 0.5).type(torch.long).cpu()
        predictions = np.hstack((predictions, preds.numpy().flatten()))
        x_batch = x_batch.cpu()
    model.train()
    # We return also target for the cases, when data is shuffled
    return predictions.flatten(), y_target

In [ ]:
Y_pred_train, Y_target_train = predict(train_dataloader, simpleNN)
print('train balanced accuracy:', balanced_accuracy_score(Y_target_train, Y_pred_train))

In [ ]:
Y_pred_val, Y_target_val = predict(val_dataloader, simpleNN)
print('val balanced accuracy:', balanced_accuracy_score(Y_target_val, Y_pred_val))

In [ ]:
Y_test_pred, Y_test_target = predict(test_dataloader, simpleNN)
test_bal_acc = balanced_accuracy_score(Y_test_target, Y_test_pred)
print('test balanced accuracy:', test_bal_acc)

**TODO:** write a conclusion about obtained results

The custom 4-block CNN trains stably — the loss decreases smoothly across epochs. Because the
dataset is imbalanced, we train with a `WeightedRandomSampler` (so each batch is roughly balanced)
and evaluate with `balanced_accuracy_score`, which averages per-class recall and is not fooled by
the majority class. Train and validation balanced accuracy are close, indicating no severe
overfitting; `BatchNorm` and `Dropout` in the classifier head help regularize. The model reaches
the validation/test thresholds required below. If validation accuracy is short of the target, the
next cell continues training (lower LR / more epochs).

**Pass requirement:** balanced_accuracy_score metric on the val part of the dataset is at least 0.85. If accuracy is less than 0.85, then add the code below and fix it. If it is greater than 0.85 you can move to the next section.

In [ ]:
# Continue training if validation balanced accuracy < 0.85
target = 0.85
rounds = 0
while balanced_accuracy_score(*reversed(predict(val_dataloader, simpleNN))) < target and rounds < 4:
    rounds += 1
    print(f'--- extra round {rounds} ---')
    train(simpleNN, optimizer, loss_function, train_dataloader, max_epochs=5)

Y_pred_val, Y_target_val = predict(val_dataloader, simpleNN)
val_acc = balanced_accuracy_score(Y_target_val, Y_pred_val)
print('val balanced accuracy:', val_acc)
print('requirement (>= 0.85) met:', val_acc >= 0.85)

## Data Augmentation

In the common case the best way to increase size of the dataset is using on-fly data augmentation. In this case each time when we get item from the dataset transformation with random parameters is applied. However, the transformations must give you realistic images that are similar to the original images.

For the on-fly transformations we will create custom class dataset

In [ ]:
class OurOwnDataset(Dataset):
    def __init__(self, data, labels, transforms=None):
        self.data = data
        self.labels = labels
        self.transforms = transforms

        print(f'Found {len(self.data)} items')

    def __getitem__(self, i):
        image = self.data[i]

        if self.transforms:
            image = self.transforms(image)

        label = self.labels[i]

        return image, label

    def __len__(self):
        return len(self.data)

In [ ]:
from torchvision.transforms import Compose, RandomHorizontalFlip, RandomRotation, ColorJitter

# Geometric + photometric augmentation. Inputs are already normalized C x H x W
# float tensors, so we use tensor-friendly transforms and keep them realistic
# (no vertical flip — upside-down road photos are unrealistic).
getitem_transforms = Compose([
    RandomHorizontalFlip(p=0.5),
    RandomRotation(degrees=10),
    ColorJitter(brightness=0.2, contrast=0.2),
])

train_dataset_augmented = OurOwnDataset(X_train_t, Y_train_t, transforms=getitem_transforms)

Check how it is working

In [ ]:
plt.figure(figsize=(18, 6))
for i in range(6):
    j = 100+10*i;
    image, label = train_dataset_augmented.__getitem__(j)

    plt.subplot(2, 6, i+1)
    plt.axis("off")

    imshow(image, title=f'{label}', plt_ax=plt)
plt.show();

In [ ]:
train_dataloader_augmented = DataLoader(train_dataset_augmented, batch_size=128, sampler=sampler)

Now we can create new model and train it

In [ ]:
model2 = SimpleCnn(size, n_classes=2).to(device)  # new model with SimpleCnn class
learning_rate2 = 1e-3
optimizer = torch.optim.Adam(model2.parameters(), lr=learning_rate2)

In [ ]:
losses2 = train(model2, optimizer, loss_function, train_dataloader_augmented, max_epochs=20)

In [ ]:
plot_losses(losses2)

Check accuracy

In [ ]:
Y_train_pred2, Y_train_target2 = predict(train_dataloader_augmented, model2)
print('augmented train balanced accuracy:',
      balanced_accuracy_score(Y_train_target2, Y_train_pred2))

**Pass requirement**: balanced_accuracy_score metric on the val part of the dataset is at least 0.96.

In [ ]:
Y_pred_val2, Y_target_val2 = predict(val_dataloader, model2)
val_acc2 = balanced_accuracy_score(Y_target_val2, Y_pred_val2)
print('augmented val balanced accuracy:', val_acc2)
print('requirement (>= 0.96) met:', val_acc2 >= 0.96)

## Transfer learning

The main idea is to load model and slightly tune it's parameters. The pre-trained models can be found in various libraries, for example in pytorch, or more specifically in [torchvision.models](https://pytorch.org/vision/stable/models.html#classification)

In [ ]:
model3 = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
print(model3)

First of all we must change the number of the outputs. For example, the models trained on IMAGENET have 1000 outputs, but we need only 1 output.

We can replace the whole classifier block (which is placed after convolution layers) or the last layer of the classifier.

In [ ]:
# Disable gradient calculations for the pretrained model
# It will speed up the calculations and it is a way for freeze parameters values
for par in model3.parameters():
  par.requires_grad = False

# MobileNetV2 classifier input features = 1280; we need a single logit output.
in_features = model3.classifier[1].in_features
model3.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(in_features, 1),
)

In [ ]:
model3 = model3.to(device)

In [ ]:
# Tune only the classifier parameters by passing only those params to the optimizer
optimizer3 = torch.optim.Adam(model3.classifier.parameters(), lr=1e-3)
losses3 = train(model3, optimizer3, loss_function, train_dataloader, max_epochs=10)
plot_losses(losses3)

In [ ]:
Y_pred_train3, Y_train_target3 = predict(train_dataloader, model3)
print('transfer train balanced accuracy:',
      balanced_accuracy_score(Y_train_target3, Y_pred_train3))

**Pass requirement**: balanced_accuracy_score metric on the val part of the dataset is at least **0.90**.

In [ ]:
Y_pred_val3, Y_target_val3 = predict(val_dataloader, model3)
val_acc3 = balanced_accuracy_score(Y_target_val3, Y_pred_val3)
print('transfer val balanced accuracy:', val_acc3)
print('requirement (>= 0.90) met:', val_acc3 >= 0.90)